# EcoData · S9 — Observabilidad con Grafana y costos (dimensión clima)
**Alahin Reyme Ticona Veliz** · Proyecto Sello EcoData (dimensión **clima / temperatura**, Juliaca) · Big Data, UPeU

La guía instrumenta un pipeline de streaming con métricas de **infraestructura** (Kafka → kafka-exporter → Prometheus → Grafana) y de **aplicación** (campos calculados en Spark). EcoData ya tiene Prometheus, Grafana, Pushgateway y 8 alertas en producción (S12); este cuaderno genera la evidencia que pide S9 sobre esa misma infraestructura, con el flujo de **clima**.

| Guía S9 | EcoData · clima |
|---|---|
| topic `atmos-eventos` | **`ecodata-clima-s09`** (lecturas reales de temperatura/humedad/viento de Juliaca, Open-Meteo) |
| grupo `uso-atmos-group` | **`ec-helada-ms-group`** y **`validador-clima-group`** (creados en S6 y S7) |
| `isValid`, `processedAt`, `latencyMs` | iguales, calculados en Spark; `isValid` aplica los **rangos físicos de S7**; `latencyMs` = proceso − **llegada a Kafka** |
| alertas propuestas (Tabla 8) | las reglas **ya implementadas** en `alerts.yml` (`ClimaSinEventosNuevos`, `ClimaModeloNoSuperaPersistencia`, `HeladaPrevista`…), más umbrales nuevos calculados con datos medidos |
| costos (Tabla 9) | plantilla con tus valores medidos y el costo de `operacion.md` |

> **Antes de empezar:** Kafka, Prometheus (49090), Grafana (43000) y Pushgateway deben estar levantados; haz S6 y S7 antes (crean los grupos de consumo que se ven en el lag). Cierra otros cuadernos de Spark (Kernel → Shut down).
>
> **Transparencia:** los eventos son horas reales de la API publicadas **a ritmo controlado** (1 cada 2 s) para poder observar latencia y rendimiento; no son un sensor en vivo.

## 0. Preparación

In [1]:
import sys, json, time, threading, urllib.request, urllib.parse, os
from datetime import datetime, timezone
import pandas as pd
sys.path.insert(0, "/tmp/s12/scripts")
import clima_comun as clima
import openmeteo_clima_a_kafka as pub_clima
from kafka import KafkaProducer, KafkaConsumer, TopicPartition
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

BOOTSTRAP = "kafka:9092"
PUSHGATEWAY = "http://pushgateway:9091"
PROMETHEUS_CANDIDATOS = ["http://prometheus:9090", "http://lambda26-prometheus:9090"]
PKG = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
HISTORICO = "/tmp/s12/clima_juliaca_2023.json"      # temperatura, humedad y viento horarios reales (Open-Meteo), ya en el proyecto

def crear_topic(nombre, particiones=3):
    adm = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
    try: adm.create_topics([NewTopic(nombre, num_partitions=particiones, replication_factor=1)]); print("topic creado:", nombre)
    except TopicAlreadyExistsError: print("el topic ya existía:", nombre)
    adm.close()

def prom(q):
    for base in PROMETHEUS_CANDIDATOS:
        try:
            u = f"{base}/api/v1/query?" + urllib.parse.urlencode({"query": q})
            return json.load(urllib.request.urlopen(u, timeout=4))["data"]["result"]
        except Exception:
            continue
    return None

def publicar_paced(topic, eventos, pausa_s=2.0):
    """Publica un evento cada pausa_s segundos en segundo plano (la celda no se bloquea)."""
    def run():
        p = KafkaProducer(bootstrap_servers=BOOTSTRAP, acks="all", key_serializer=str.encode,
                          value_serializer=lambda v: v if isinstance(v, bytes) else json.dumps(v).encode())
        for e in eventos:
            p.send(topic, key=e["sensorId"], value=e); p.flush(); time.sleep(pausa_s)
        p.close()
    th = threading.Thread(target=run, daemon=True); th.start(); return th

def desde_archivo(path=HISTORICO):
    """[(ts_ms, temp, hum, viento)] ordenado y sin huecos, del histórico real del proyecto."""
    return clima.leer_json(path)

def serie_clima(min_horas=96):
    """Primero la API (horas recientes); si trae pocas (suele dar ~50 h), el histórico real. Devuelve (serie, origen)."""
    try:
        s = pub_clima.desde_api(); origen = "openmeteo-api"
    except Exception as ex:
        print("API no disponible:", type(ex).__name__); s, origen = [], "openmeteo-api"
    if len(s) < min_horas:
        print(f"La API trajo {len(s)} horas; uso el histórico real de Juliaca (origen openmeteo-replay)")
        s, origen = desde_archivo(), "openmeteo-replay"
    return s, origen

TOPIC = "ecodata-clima-s09"
crear_topic(TOPIC)

topic creado: ecodata-clima-s09


## 1. Prometheus: targets y consultas PromQL (3.1–3.2)
Verifica a mano en http://localhost:49090 → **Status → Targets** (`kafka-exporter`, `pushgateway` y `prometheus` en **UP**) y captura la pantalla. Luego corre las consultas desde aquí:

In [2]:
consultas = {
  "kafka_brokers": "kafka_brokers",
  "exporter arriba": 'up{job="kafka-exporter"}',
  "pushgateway arriba": 'up{job="pushgateway"}',
  "lag ec-helada-ms-group": 'sum(kafka_consumergroup_lag{consumergroup="ec-helada-ms-group"})',
  "lag validador-clima-group": 'sum(kafka_consumergroup_lag{consumergroup="validador-clima-group"})',
  "mensajes en atmos-clima": 'sum(kafka_topic_partition_current_offset{topic="atmos-clima"})',
}
for nombre, q in consultas.items():
    r = prom(q)
    print(f"{nombre:30s}", "SIN RESPUESTA de Prometheus" if r is None else (r[0]["value"][1] if r else "sin datos (¿el grupo ya existe? corre S6/S7)"))

kafka_brokers                  1
exporter arriba                1
pushgateway arriba             1
lag ec-helada-ms-group         0
lag validador-clima-group      0
mensajes en atmos-clima        13591


**Tablero de Grafana (3.3):** importa `s09_dashboard_clima.json` (Dashboards → New → Import → Upload JSON file, datasource Prometheus). Trae los cuatro paneles de la guía (Kafka Brokers, Exporter Up, Broker Info, Consumer Lag) más lag por grupo y la frescura del job de clima.

## 2. Spark: lectura streaming con campos de observabilidad (3.4–3.5)
`isValid` reutiliza el contrato mínimo **y los rangos físicos de S7**; `processedAt` es el instante en que Spark procesa; `latencyMs = processedAt − kafkaTimestamp` mide cuánto tardó el evento desde que llegó al broker hasta Spark (el `timestamp` del evento es la **hora del dato** y puede ser de horas atrás, por eso no sirve para medir latencia del pipeline).

In [3]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StringType, DoubleType, LongType
esquema_evento = (StructType().add("tipoEvento", StringType()).add("sensorId", StringType())
                  .add("temp", DoubleType()).add("hum", DoubleType()).add("viento", DoubleType())
                  .add("origen", StringType()).add("timestamp", LongType()))
spark = (SparkSession.builder.appName("ecodata-clima-s09").master("local[2]")
         .config("spark.jars.packages", PKG)
         .config("spark.sql.shuffle.partitions", "3")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.port", "4060")          # puertos 4060-4063: no chocan con 4040 (S12) ni con 4050-4053 (aire)
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3af16ad3-72be-4e11-aef9-15eb449c7cb4;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in spark-list
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.

Spark 4.2.0


In [4]:
crudo = (spark.readStream.format("kafka").option("kafka.bootstrap.servers", BOOTSTRAP)
         .option("subscribe", "ecodata-clima-s09").option("startingOffsets", "latest").option("failOnDataLoss", "false").load())
eventos = (crudo.select("topic", "partition", "offset", F.col("timestamp").alias("kafkaTimestamp"),
                        F.from_json(F.col("value").cast("string"), esquema_evento).alias("e"))
           .select("topic", "partition", "offset", "kafkaTimestamp", "e.*"))
observables = (eventos
    .withColumn("isValid", F.col("tipoEvento").isNotNull() & F.col("sensorId").isNotNull() & F.col("timestamp").isNotNull()
                           & F.col("temp").between(-30, 35) & F.col("hum").between(0, 100) & (F.col("viento") >= 0))
    .withColumn("processedAt", F.unix_millis(F.current_timestamp()))
    .withColumn("latencyMs", F.col("processedAt") - F.unix_millis(F.col("kafkaTimestamp"))))
observables.printSchema()

root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTimestamp: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- temp: double (nullable = true)
 |-- hum: double (nullable = true)
 |-- viento: double (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- isValid: boolean (nullable = true)
 |-- processedAt: long (nullable = false)
 |-- latencyMs: long (nullable = true)



## 3. Salida dual: Parquet + bitácora por micro-lote (3.6 y 3.8)
Cada micro-lote se guarda en Parquet y deja una línea en una bitácora con el formato de logs de la guía. (La consola de Spark no se ve en la celda de Jupyter, por eso la bitácora la reemplaza.)

In [5]:
RUTA = "./artifacts/s09_clima_observabilidad"; CHK = "./artifacts/chk_s09_clima"
bitacora = []
def por_microlote(df, bid):
    n = df.count()
    if n == 0:
        bitacora.append(dict(service="spark-streaming", component="observabilidad-clima", batchId=bid, numInputRows=0, status="idle")); return
    df.write.mode("append").parquet(RUTA)
    r = df.agg(F.avg("latencyMs").alias("lat"), F.sum((~F.col("isValid")).cast("int")).alias("inval")).first()
    bitacora.append(dict(service="spark-streaming", component="observabilidad-clima", topic="ecodata-clima-s09", batchId=bid,
                         numInputRows=n, avgLatencyMs=round(r["lat"], 1), invalidos=int(r["inval"]), status="processed"))

consulta = (observables.writeStream.foreachBatch(por_microlote)
            .option("checkpointLocation", CHK).trigger(processingTime="5 seconds").start())
print("consulta activa:", consulta.isActive)

consulta activa: True


Con la consulta activa, **publica el flujo** (45 lecturas reales, una cada 2 s ≈ 90 s). No detengas la consulta todavía. Se toma el tramo más frío del histórico si la API trae pocas horas.

In [6]:
serie, ORIGEN = serie_clima(48)
if ORIGEN == "openmeteo-replay":
    i_min = min(range(len(serie)), key=lambda i: serie[i][1]); ini = max(0, min(i_min - 24, len(serie) - 48)); serie = serie[ini:ini + 48]
paced = [clima.evento(t, x, h, v, ORIGEN) for t, x, h, v in (serie * 3)[-45:]]
hilo = publicar_paced(TOPIC, paced, pausa_s=2.0)
print(len(paced), "eventos en camino hacia", TOPIC, "· origen:", ORIGEN)

45 eventos en camino hacia ecodata-clima-s09 · origen: openmeteo-api


## 4. lastProgress: throughput y lag del lado de Spark (3.7)
Espera ~40 s desde la celda anterior y corre esta celda **dos o tres veces**.

In [7]:
p = consulta.lastProgress
if p is None:
    print("Todavía no hay micro-lote procesado. Espera unos segundos y repite.")
else:
    m = p["sources"][0].get("metrics", {})
    print(json.dumps({"service": "spark-streaming", "component": "observabilidad-clima", "topic": "ecodata-clima-s09",
        "batchId": p["batchId"], "numInputRows": p["numInputRows"], "inputRowsPerSecond": p["inputRowsPerSecond"],
        "processedRowsPerSecond": p["processedRowsPerSecond"], "avgOffsetsBehindLatest": m.get("avgOffsetsBehindLatest"),
        "maxOffsetsBehindLatest": m.get("maxOffsetsBehindLatest"),
        "status": "idle" if p["numInputRows"] == 0 else "processed"}, indent=2))
print("\nÚltimas líneas de la bitácora:")
for b in bitacora[-5:]: print(json.dumps(b))

{
  "service": "spark-streaming",
  "component": "observabilidad-clima",
  "topic": "ecodata-clima-s09",
  "batchId": 4,
  "numInputRows": 9,
  "inputRowsPerSecond": 1.8,
  "processedRowsPerSecond": 4.2,
  "avgOffsetsBehindLatest": "0.0",
  "maxOffsetsBehindLatest": "0",
  "status": "processed"
}

Últimas líneas de la bitácora:
{"service": "spark-streaming", "component": "observabilidad-clima", "batchId": 0, "numInputRows": 0, "status": "idle"}
{"service": "spark-streaming", "component": "observabilidad-clima", "topic": "ecodata-clima-s09", "batchId": 1, "numInputRows": 2, "avgLatencyMs": 2099.5, "invalidos": 0, "status": "processed"}
{"service": "spark-streaming", "component": "observabilidad-clima", "topic": "ecodata-clima-s09", "batchId": 2, "numInputRows": 3, "avgLatencyMs": 2088.7, "invalidos": 0, "status": "processed"}
{"service": "spark-streaming", "component": "observabilidad-clima", "topic": "ecodata-clima-s09", "batchId": 3, "numInputRows": 2, "avgLatencyMs": 2078.5, "invalid

Evidencia alternativa: **Spark UI** en http://localhost:4060 → *Structured Streaming* (el puerto 4060 evita chocar con los jobs de S12, que usan 4040; si no abre desde tu navegador, usa el bloque de arriba).

In [8]:
hilo.join(timeout=120)          # espera a que termine la publicación
time.sleep(12)                  # y a que Spark procese el último micro-lote
consulta.stop()
guardado = spark.read.parquet(RUTA)
print("filas guardadas:", guardado.count())
guardado.select("partition", "offset", "sensorId", "temp", "hum", "isValid", "latencyMs").orderBy(F.desc("offset")).show(10, truncate=False)

filas guardadas: 45
+---------+------+-----------------------+----+----+-------+---------+
|partition|offset|sensorId               |temp|hum |isValid|latencyMs|
+---------+------+-----------------------+----+----+-------+---------+
|2        |44    |openmeteo-clima-juliaca|8.7 |75.0|true   |4914     |
|2        |43    |openmeteo-clima-juliaca|8.6 |72.0|true   |1918     |
|2        |42    |openmeteo-clima-juliaca|9.0 |70.0|true   |3923     |
|2        |41    |openmeteo-clima-juliaca|9.8 |69.0|true   |928      |
|2        |40    |openmeteo-clima-juliaca|12.3|52.0|true   |2931     |
|2        |39    |openmeteo-clima-juliaca|14.6|36.0|true   |4935     |
|2        |38    |openmeteo-clima-juliaca|16.0|31.0|true   |1940     |
|2        |37    |openmeteo-clima-juliaca|17.4|27.0|true   |3944     |
|2        |36    |openmeteo-clima-juliaca|18.1|26.0|true   |947      |
|2        |35    |openmeteo-clima-juliaca|18.8|22.0|true   |2950     |
+---------+------+-----------------------+----+----+-----

## 5. Latencia agregada por ventana de tiempo (3.9)

In [9]:
lat = (spark.read.parquet(RUTA).groupBy(F.window("kafkaTimestamp", "1 minute"))
       .agg(F.count("*").alias("numEventos"), F.avg("latencyMs").alias("avgLatencyMs"),
            F.min("latencyMs").alias("minLatencyMs"), F.max("latencyMs").alias("maxLatencyMs"))
       .select(F.col("window.start").alias("inicioVentana"), "numEventos", "avgLatencyMs", "minLatencyMs", "maxLatencyMs")
       .orderBy("inicioVentana"))
lat.show(truncate=False)
EXPLICACION = ("El productor envía cada 2 s y Spark dispara cada 5 s: no son múltiplos (mcm = 10 s), así que la espera de "
               "cada evento hasta el siguiente disparo cicla entre ~0 y ~5 s. Esa es la variación de min/max: aritmética, no una falla.")
print(EXPLICACION)

+-------------------+----------+-----------------+------------+------------+
|inicioVentana      |numEventos|avgLatencyMs     |minLatencyMs|maxLatencyMs|
+-------------------+----------+-----------------+------------+------------+
|2026-11-06 04:11:00|22        |2055.5           |19          |4093        |
|2026-11-06 04:12:00|23        |3045.695652173913|928         |4995        |
+-------------------+----------+-----------------+------------+------------+

El productor envía cada 2 s y Spark dispara cada 5 s: no son múltiplos (mcm = 10 s), así que la espera de cada evento hasta el siguiente disparo cicla entre ~0 y ~5 s. Esa es la variación de min/max: aritmética, no una falla.


## 6. Alertas con umbrales justificados por datos medidos (3.10)

In [10]:
pl = spark.read.parquet(RUTA).select("latencyMs").toPandas()["latencyMs"]
p50, p95, mx = pl.quantile(.50), pl.quantile(.95), pl.max()
u_sens, u_crit = int(round(2 * p95, -2)), int(round(10 * p95, -2))
print(f"latencia medida: p50={p50:.0f} ms · p95={p95:.0f} ms · máx={mx:.0f} ms")
alertas = pd.DataFrame([
  ["Kafka no visible", "kafka_brokers < 1", "Prometheus", "propuesta"],
  ["Exporter caído", 'up{job="kafka-exporter"} == 0', "Prometheus", "propuesta"],
  ["Lag alto sostenido", 'sum(kafka_consumergroup_lag{consumergroup="ec-helada-ms-group"}) > 100 (for 2m)', "Prometheus", "propuesta; el lag normal medido en S6 vuelve a 0"],
  ["Latencia sensible", f"avgLatencyMs por minuto > {u_sens} ms (2×p95 medido)", "Spark / Parquet", "propuesta con tu p95"],
  ["Latencia crítica", f"avgLatencyMs por minuto > {u_crit} ms (10×p95 medido)", "Spark / Parquet", "propuesta con tu p95"],
  ["Sin eventos de clima nuevos", "time() - push_time_seconds{job=lambda26_clima} > 600  or absent(...)", "Prometheus", "IMPLEMENTADA (alerts.yml): ClimaSinEventosNuevos"],
  ["Modelo de temperatura no supera persistencia", "lambda26_clima_mejora_pct < 0 (10 m)", "Prometheus", "IMPLEMENTADA: ClimaModeloNoSuperaPersistencia"],
  ["Helada prevista", "temperatura prevista < 0 °C", "Prometheus", "IMPLEMENTADA: HeladaPrevista"],
], columns=["Situación", "Regla", "Dónde se evalúa", "Estado"])
alertas

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


latencia medida: p50=2931 ms · p95=4949 ms · máx=4995 ms


,Situación,Regla,Dónde se evalúa,Estado
0,Kafka no visible,kafka_brokers < 1,Prometheus,propuesta
1,Exporter caído,"up{job=""kafka-exporter""} == 0",Prometheus,propuesta
2,Lag alto sostenido,"sum(kafka_consumergroup_lag{consumergroup=""ec-...",Prometheus,propuesta; el lag normal medido en S6 vuelve a 0
3,Latencia sensible,avgLatencyMs por minuto > 9900 ms (2×p95 medido),Spark / Parquet,propuesta con tu p95
4,Latencia crítica,avgLatencyMs por minuto > 49500 ms (10×p95 med...,Spark / Parquet,propuesta con tu p95
5,Sin eventos de clima nuevos,time() - push_time_seconds{job=lambda26_clima}...,Prometheus,IMPLEMENTADA (alerts.yml): ClimaSinEventosNuevos
6,Modelo de temperatura no supera persistencia,lambda26_clima_mejora_pct < 0 (10 m),Prometheus,IMPLEMENTADA: ClimaModeloNoSuperaPersistencia
7,Helada prevista,temperatura prevista < 0 °C,Prometheus,IMPLEMENTADA: HeladaPrevista


## 7. Costos y plan de escalado (3.11)

In [14]:
import glob
bytes_parquet = sum(os.path.getsize(f) for f in glob.glob(RUTA + "/*.parquet"))
filas_parquet = spark.read.parquet(RUTA).count()
b_por_fila = bytes_parquet / max(filas_parquet, 1)
_t = spark.read.parquet(RUTA).agg(F.min("kafkaTimestamp").alias("a"), F.max("kafkaTimestamp").alias("b")).first()
ritmo_lab = (filas_parquet - 1) / max((_t["b"] - _t["a"]).total_seconds(), 1)   # medido: filas guardadas / tiempo entre primer y último evento
ritmo_real = 1 / 3600                            # clima real: 1 evento/hora (temperatura, humedad y viento van en el mismo evento)
def gb_mes(ev_s): return ev_s * b_por_fila * 3600 * 24 * 30 / 1e9
EC2_HORA, S3_GB_MES, INSTANCIAS = 0.0416, 0.023, 3     # t3.medium bajo demanda y S3 Standard (referencia de la guía; verifica el precio vigente)
computo = INSTANCIAS * EC2_HORA * 730
tabla = pd.DataFrame([
  ["Eventos por segundo (laboratorio, medido)", f"{ritmo_lab:.2f}", "filas en Parquet / tiempo entre primer y último evento (celda 8)"],
  ["Eventos por segundo (clima real)", f"{ritmo_real:.5f}", "1 evento/hora"],
  ["Latencia promedio por minuto (medida)", f"{pl.mean():.0f} ms", "celda 5"],
  ["Bytes por fila en Parquet", f"{b_por_fila:.0f} B", f"{bytes_parquet:,} B / {filas_parquet} filas"],
  ["Crecimiento Parquet, laboratorio", f"{gb_mes(ritmo_lab):.2f} GB/mes", "si corriera 24/7 a este ritmo"],
  ["Crecimiento Parquet, clima real", f"{gb_mes(ritmo_real)*1000:.2f} MB/mes", "ritmo real"],
  ["Costo cómputo (3 instancias t3.medium)", f"US$ {computo:.2f}/mes", "3 × 0,0416 × 730 h (Kafka; Spark; Prometheus+Grafana)"],
  ["Costo almacenamiento (laboratorio)", f"US$ {gb_mes(ritmo_lab)*S3_GB_MES:.2f}/mes", "S3 Standard 0,023 US$/GB-mes"],
], columns=["Variable", "Valor medido / calculado", "Origen"])
tabla

,Variable,Valor medido / calculado,Origen
0,"Eventos por segundo (laboratorio, medido)",0.50,filas en Parquet / tiempo entre primer y últim...
1,Eventos por segundo (clima real),0.00028,1 evento/hora
2,Latencia promedio por minuto (medida),2562 ms,celda 5
3,Bytes por fila en Parquet,1845 B,"83,017 B / 45 filas"
4,"Crecimiento Parquet, laboratorio",2.39 GB/mes,si corriera 24/7 a este ritmo
5,"Crecimiento Parquet, clima real",1.33 MB/mes,ritmo real
6,Costo cómputo (3 instancias t3.medium),US$ 91.10/mes,"3 × 0,0416 × 730 h (Kafka; Spark; Prometheus+G..."
7,Costo almacenamiento (laboratorio),US$ 0.05/mes,"S3 Standard 0,023 US$/GB-mes"


In [12]:
prog = [p for p in consulta.recentProgress if p["numInputRows"] > 0]
filas_leidas = sum(p["numInputRows"] for p in prog)
ritmo_medido = sum(p["inputRowsPerSecond"] for p in prog) / max(len(prog), 1)
print("lotes con datos:", len(prog), "| filas leídas:", filas_leidas, "| ritmo medio:", round(ritmo_medido, 2), "ev/s")

lotes con datos: 19 | filas leídas: 135 | ritmo medio: 1.42 ev/s


numInputRows de Spark suma 135 filas frente a 45 guardadas porque foreachBatch evalúa el DataFrame tres veces (count, write, agg). El ritmo real del laboratorio se calcula con las filas guardadas (0,50 ev/s).

**Plan de escalado — escribe tu decisión con estos números** (ejemplo de estructura):
1. *Umbral 1:* si `kafka_consumergroup_lag` > 100 de forma sostenida (> 2 min) → subir particiones y consumidores.
2. *Umbral 2:* si la latencia promedio por minuto supera el valor crítico de la celda 6 → más paralelismo en Spark.
3. *Si el volumen se duplicara:* con el ritmo real del clima (1 evento/hora) el cuello de botella no es el broker sino el modelo/Spark; se escalaría primero Spark. Justifícalo con tus medidas.

### Qué capturar (con **reloj del sistema y tu perfil visibles, sin recortar**)
1. Prometheus → Targets (UP) y las consultas de la sección 1.
2. Grafana: el tablero de S9 con los cuatro paneles.
3. Sección 2 (esquema con `isValid/processedAt/latencyMs`) y sección 4 (`lastProgress` + bitácora).
4. Sección 5 (latencia por minuto) y sección 6 (tabla de alertas).
5. Sección 7 (plantilla de costos) y tu texto del plan de escalado.

In [15]:
spark.stop()